In [2]:
from analyze_video.utils import *
import os
from tqdm import tqdm

In [ ]:
# Downloaded videos are in /home/tyler/python/agents/raw_videos/penguin
# Subtitles for those videos in srt format are in /home/tyler/python/agents/raw_video_transcripts/penguin

In [4]:
RAW_VIDEO_DIR = "/home/tyler/python/agents/raw_videos/penguin/"
TRANSCRIPTS_FOLDER = "/home/tyler/python/agents/raw_video_transcripts/penguin/"

In [1]:
from datasets import load_dataset

dataset = load_dataset("mlabonne/FineTome-100k", split="train[:3000]")

/home/tyler/python/agents/agents/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 100000/100000 [00:00<00:00, 209682.59 examples/s]


In [ ]:
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to /home/tyler/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
def convert_srt_to_alpaca(srt_path):

    with open(srt_path, "r") as f:
        srt_content = f.read()

    lines = srt_content.split("\n")
    lines = [lines[i] for i in range(2, len(lines), 4)]

    lines = " ".join(lines)
    lines = lines.split(".")

    outputs = []
    total_chunks = (len(lines) + 3) // 4

    for i in tqdm(
        range(0, len(lines), 4), desc="Processing chunks", total=total_chunks
    ):
        input_sentences = lines[i : i + 4]
        input_sentences = " \n".join(input_sentences)
        response = prompt_ollama(
            prompt=f"Re-write the following sentence by removing the author's tone and as if it was a wikipedia article, with only the facts stated. Do not return anything else. \n\n {input_sentences}"
        )
        dict_unit = {
            "instruction": "Re-write the following in the tone of x0rv",
            "input": response,
            "output": input_sentences,
        }
        outputs.append(dict_unit)
    return outputs


big_json = []
for srt_file in os.listdir(TRANSCRIPTS_FOLDER):
    output = convert_srt_to_alpaca(os.path.join(TRANSCRIPTS_FOLDER, srt_file))
    big_json = big_json + output

  0%|          | 0/22 [04:51<?, ?it/s]


KeyboardInterrupt: 

In [7]:
test_json

[{'instruction': "Re-write the following sentence by removing the author's tone and as if it was a wikipedia article, with only the facts stated. Do not return anything else. \n\n ",
  'input': "In my quest to watch every video game movie ever made, I just made it to one that I was really \ndreading. And boy howdy did it hit me like a brick. A punishment I wouldn't wish upon my worst \nenemy. Watching the Assassin's Creed movie. I have a theory that most people that watch this film \nin theaters dumped it from their memories as a way of coping with the traumatic experience of",
  'output': "The Assassin's Creed movie has been noted by some viewers to be poorly received, with opinions suggesting that audience members may deliberately forget the film after viewing it in an effort to cope with its perceived negative impact."},
 {'instruction': "Re-write the following sentence by removing the author's tone and as if it was a wikipedia article, with only the facts stated. Do not return anyt